In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/Users/sakshamchauhan/Downloads/Github/Data_Science/Deep_Learning/Lecture-3/qoute_dataset.csv")

In [3]:
df.head(5)

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [5]:
df.shape

(3038, 2)

In [6]:
quotes = df['quote']
quotes.head(5)

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: str

In [7]:
quotes = quotes.str.lower()

In [8]:
import string
translator = str.maketrans('','',string.punctuation)
quotes =quotes.apply(lambda x:x.translate(translator))

In [9]:
quotes.head(5)

0    “the world as we have created it is a process ...
1    “it is our choices harry that show what we tru...
2    “there are only two ways to live your life one...
3    “the person be it gentleman or lady who has no...
4    “imperfection is beauty madness is genius and ...
Name: quote, dtype: str

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [11]:
vocab_size = 10000

tokenizer = Tokenizer(num_words= vocab_size)
tokenizer.fit_on_texts(quotes)

In [12]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [13]:
sequence = tokenizer.texts_to_sequences(quotes)

In [14]:
for i in range(3):
    print(quotes[i])
print()
for i in range(3):
    print(sequence[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [15]:
x = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        x.append(input_seq)
        y.append(output_seq)

In [16]:
len(x)

85271

In [17]:
len(y)

85271

In [18]:
max_len = max(len(x) for x in x)
print(max_len)

745


In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
x_padded = pad_sequences(x, maxlen=max_len, padding= 'pre')

In [20]:
y = np.array(y)

In [21]:
x_padded.shape

(85271, 745)

In [22]:
from tensorflow.keras.utils import to_categorical
y_ohe = to_categorical(y, num_classes = vocab_size)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [24]:
embed_dim = 50
rnn_units = 128

In [ ]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim= vocab_size, output_dim= embed_dim, input_length= max_len)
)

rnn_model.add(SimpleRNN(units = rnn_units))
rnn_model.add(Dense(units = vocab_size, activation= 'softmax'))

/Users/sakshamchauhan/miniconda3/envs/tensorflow/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-09-03 11:27:40.956587: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-09-03 11:27:40.956880: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-03 11:27:40.956905: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
I0000 00:00:1788415060.957742 3578865 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1788415060.958376 3578865 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [ ]:
rnn_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [27]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
lstm_model = Sequential()

lstm_model.add(
    Embedding(input_dim= vocab_size, output_dim= embed_dim, input_length= max_len)
)

lstm_model.add(LSTM(units = rnn_units))
lstm_model.add(Dense(units = vocab_size, activation= 'softmax'))

/Users/sakshamchauhan/miniconda3/envs/tensorflow/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [29]:
lstm_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [30]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
epochs = 2
batch_size = 128

In [33]:
history_rnn = rnn_model.fit(
    x_padded, y_ohe,
    epochs = epochs,
    batch_size = batch_size,
    validation_split = 0.2
)

: 